Cell 1: Initialization & Configurations

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Define Paths & Bucket 
BUCKET = "ecommerce-youssef"
RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"
PROCESSED_PATH = f"s3://{BUCKET}/processed/ecommerce"

print(" Configuration completed for Bucket:", BUCKET)

 Configuration completed for Bucket: ecommerce-youssef


Cell 2: Part 1 - Read & Profile Data

In [8]:
# 1. read files
customers = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/customers.csv")
categories = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/categories.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/products.csv")
departments = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/departments.csv")
employees = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/employees.csv")
suppliers = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/suppliers.csv")
orders = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/orders.csv")
order_details = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/order_details.csv")
payments = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/payments.csv")
product_suppliers = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/product_suppliers.csv")
shippers = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/shippers.csv")
shipments = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW_PATH}/shipments.csv")

# 2. Display schemas & 10 records
customers.printSchema()
customers.show(10, truncate=False)

# 4. Count records
print("Customers count:", customers.count())
print("Orders count:", orders.count())

# 5. Calculate NULL values in customers
customers.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in customers.columns]).show()

# 6 & 7. Duplicate customerid and orderid
print("Duplicate Customers:", customers.groupBy("CustomerID").count().filter("count > 1").count())
print("Duplicate Orders:", orders.groupBy("OrderID").count().filter("count > 1").count())

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City      |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza      |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai     |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza      |United Kingdom|2020-12-22 00:00:00|
|4         |Patricia |Miller  |patricia.miller4@example.com

Cell 3: Part 2 & Part 3 - Data Cleaning & Type Transformations

In [24]:
# 8. Standardize column names to lowercase
def clean_cols(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_").replace("-", "_"))
    return df

customers = clean_cols(customers)
categories = clean_cols(categories)
products = clean_cols(products)
departments = clean_cols(departments)
employees = clean_cols(employees)
suppliers = clean_cols(suppliers)
orders = clean_cols(orders)
order_details = clean_cols(order_details)
payments = clean_cols(payments)
product_suppliers = clean_cols(product_suppliers)
shippers = clean_cols(shippers)
shipments = clean_cols(shipments)

print("--- Part 2: Column Names Standardized ---")
print("Customers Columns:", customers.columns)
print("Orders Columns:", orders.columns)

# 9, 10, 11. Trim spaces, Lowercase email, Uppercase status
customers = customers.withColumn("email", F.lower(F.trim("email")))\
                     .withColumn("firstname", F.trim("firstname"))\
                     .withColumn("lastname", F.trim("lastname"))
orders = orders.withColumn("status", F.upper(F.trim("status")))

print("\n--- Part 2: String Formatting (Email & Status Sample) ---")
customers.select("firstname", "lastname", "email").show(5, truncate=False)
orders.select("orderid", "status").show(5, truncate=False)

# 12, 13, 14. Deduplication & Remove NULL PKs
customers = customers.dropDuplicates(["customerid"]).filter(F.col("customerid").isNotNull())
orders = orders.dropDuplicates(["orderid"]).filter(F.col("orderid").isNotNull())
products = products.dropDuplicates(["productid"]).filter(F.col("productid").isNotNull())

print("\n--- Part 2: Deduplication Counts ---")
print("Clean Customers Count:", customers.count())
print("Clean Orders Count:", orders.count())

# 15, 16, 17, 18. Validation checks
invalid_emails = customers.filter(~F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))
invalid_orders = orders.filter(~F.col("status").isin("PENDING", "SHIPPED", "DELIVERED", "CANCELLED"))
invalid_products = products.filter(F.col("price").isNull() | (F.col("price") <= 0))

print("\n--- Part 2: Data Quality & Validation Results ---")
print("Invalid Emails Count:", invalid_emails.count())
invalid_emails.select("customerid", "email").show(5, truncate=False)

print("Invalid Order Statuses Count:", invalid_orders.count())
invalid_orders.select("orderid", "status").show(5, truncate=False)

print("Invalid Products (NULL or Price <= 0) Count:", invalid_products.count())
invalid_products.select("productid", "price").show(5, truncate=False)

# 19, 20, 21, 22, 23. Data Type Transformations & Date Standardization Across All Tables

customers = customers.withColumn("registrationdate", F.to_date("registrationdate"))
shipments = shipments.withColumn("shipdate", F.to_date("shipdate"))\
                     .withColumn("deliverydate", F.to_date("deliverydate"))

customers = customers.withColumn("customerid", F.col("customerid").cast("long"))\
                     .withColumn("registrationdate", F.to_date("registrationdate"))

orders = orders.withColumn("orderid", F.col("orderid").cast("long"))\
               .withColumn("customerid", F.col("customerid").cast("long"))\
               .withColumn("orderdate", F.to_date("orderdate"))

products = products.withColumn("productid", F.col("productid").cast("long"))\
                   .withColumn("price", F.col("price").cast("decimal(12,2)"))

order_details = order_details.withColumn("quantity", F.col("quantity").cast("integer"))\
                             .withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)"))

payments = payments.withColumn("amount", F.col("amount").cast("decimal(12,2)"))\
                   .withColumn("paymentdate", F.to_date("paymentdate"))

shipments = shipments.withColumn("shipdate", F.to_date("shipdate"))\
                     .withColumn("deliverydate", F.to_date("deliverydate"))

employees = employees.withColumn("hiredate", F.to_date("hiredate"))

print("\n--- Part 3: Data Schema Transformations ---")
print("Orders Schema:")
orders.printSchema()
print("Products Schema:")
products.printSchema()

# 24. Extract date columns from orderdate
orders = orders.withColumn("year", F.year("orderdate"))\
               .withColumn("month", F.month("orderdate"))\
               .withColumn("quarter", F.quarter("orderdate"))\
               .withColumn("day", F.dayofmonth("orderdate"))\
               .withColumn("day_of_week", F.dayofweek("orderdate"))

print("\n--- Part 3: Extracted Date Columns Sample ---")
orders.select("orderid", "orderdate", "year", "month", "quarter", "day", "day_of_week").show(5, truncate=False)

print("\nData Cleaning and Type Transformations completed successfully!")

--- Part 2: Column Names Standardized ---
Customers Columns: ['customerid', 'firstname', 'lastname', 'email', 'city', 'country', 'registrationdate', 'full_name', 'total_customer_sales', 'total_cust_sales', 'customer_segment']
Orders Columns: ['orderid', 'customerid', 'orderdate', 'status', 'year', 'month', 'quarter', 'day', 'day_of_week', 'calculated_order_total', 'order_total_amount', 'order_total_quantity', 'calc_order_qty', 'calc_order_qty', 'calc_order_total', 'calc_order_qty', 'order_value_category']

--- Part 2: String Formatting (Email & Status Sample) ---
+---------+--------+-----------------------------+
|firstname|lastname|email                        |
+---------+--------+-----------------------------+
|Danielle |Johnson |danielle.johnson1@example.com|
|Joshua   |Walker  |joshua.walker2@example.com   |
|Jill     |Rhodes  |jill.rhodes3@example.com     |
|Patricia |Miller  |patricia.miller4@example.com |
|Robert   |Johnson |robert.johnson5@example.com  |
+---------+--------+--

Cell 4: Part 4 - Business Transformations

In [25]:
# 25. Calculate total_amount for each item in order_details
order_details = order_details.withColumn(
    "total_amount", 
    (F.col("quantity") * F.col("unitprice")).cast("decimal(12,2)")
)
print("=== 25. Order Details with Total Amount ===")
order_details.select("orderdetailid", "orderid", "productid", "quantity", "unitprice", "total_amount").show(5)

# 26. Create full_name in customers table
customers = customers.withColumn(
    "full_name", 
    F.concat_ws(" ", F.col("firstname"), F.col("lastname"))
)
print("=== 26. Customers with Full Name ===")
customers.select("customerid", "firstname", "lastname", "full_name").show(5)

# 29 & 30. Calculate total amount and total quantity per order (OrderID)
order_aggregates = order_details.groupBy("orderid").agg(
    F.sum("total_amount").alias("calc_order_total"),
    F.sum("quantity").alias("calc_order_qty")
)
print("=== 29 & 30. Aggregated Totals Per Order ===")
order_aggregates.show(5)

# Clean orders DataFrame if columns already exist from previous runs
if "calc_order_total" in orders.columns:
    orders = orders.drop("calc_order_total", "order_value_category")

# 27. Add order_value_category to orders table
orders = orders.join(order_aggregates, "orderid", "left").withColumn(
    "order_value_category",
    F.when(F.col("calc_order_total") >= 1000, "HIGH")
     .when((F.col("calc_order_total") >= 500) & (F.col("calc_order_total") < 1000), "MEDIUM")
     .otherwise("LOW")
)
print("=== 27. Orders with Calculated Total and Category ===")
orders.select("orderid", "customerid", "calc_order_total", "order_value_category").show(5)

# Clean customers DataFrame if columns already exist from previous runs
if "total_cust_sales" in customers.columns:
    customers = customers.drop("total_cust_sales", "customer_segment")

# 28. Calculate total sales per customer and assign customer_segment
customer_sales_agg = orders.groupBy("customerid").agg(
    F.sum("calc_order_total").alias("total_cust_sales")
)

customers = customers.join(customer_sales_agg, "customerid", "left").withColumn(
    "customer_segment",
    F.when(F.col("total_cust_sales") >= 5000, "VIP")
     .when((F.col("total_cust_sales") >= 2000) & (F.col("total_cust_sales") < 5000), "PREMIUM")
     .when((F.col("total_cust_sales") >= 500) & (F.col("total_cust_sales") < 2000), "REGULAR")
     .otherwise("LOW_VALUE")
)
print("=== 28. Customers with Total Sales and Segment ===")
customers.select("customerid", "full_name", "total_cust_sales", "customer_segment").show(5)

# 31. Calculate average product price
avg_product_price = products.select(F.avg("price")).collect()[0][0]

print("Part 4 Business Transformations Completed Successfully!")
print(f"Average Product Price: {avg_product_price:.2f}")

=== 25. Order Details with Total Amount ===
+-------------+-------+---------+--------+---------+------------+
|orderdetailid|orderid|productid|quantity|unitprice|total_amount|
+-------------+-------+---------+--------+---------+------------+
|            1|  19370|      974|       5|  2877.48|    14387.40|
|            2|  49722|      403|       3|  1355.69|     4067.07|
|            3|  36676|      882|       9|  2249.53|    20245.77|
|            4|   6008|      421|       6|  1748.31|    10489.86|
|            5|  28234|       54|       6|   453.59|     2721.54|
+-------------+-------+---------+--------+---------+------------+
only showing top 5 rows

=== 26. Customers with Full Name ===
+----------+---------+--------+----------------+
|customerid|firstname|lastname|       full_name|
+----------+---------+--------+----------------+
|         1| Danielle| Johnson|Danielle Johnson|
|         2|   Joshua|  Walker|   Joshua Walker|
|         3|     Jill|  Rhodes|     Jill Rhodes|
|     

Cell 5: Part 5 - Window Functions

In [26]:

# 32. Find the latest order for every customer
cust_window_desc = Window.partitionBy("customerid").orderBy(F.col("orderdate").desc(), F.col("orderid").desc())
latest_orders = orders.withColumn("rn", F.row_number().over(cust_window_desc)).filter("rn = 1").drop("rn")
print("=== 32. Latest Order Per Customer ===")
latest_orders.select("customerid", "orderid", "status", "orderdate").show(10, truncate=False)

# 33. Find the first order for every customer
cust_window_asc = Window.partitionBy("customerid").orderBy(F.col("orderdate").asc(), F.col("orderid").asc())
first_orders = orders.withColumn("rn", F.row_number().over(cust_window_asc)).filter("rn = 1").drop("rn")
print("=== 33. First Order Per Customer ===")
first_orders.select("customerid", "orderid", "status", "orderdate").show(10, truncate=False)

# 34. Rank customers by total sales
ranked_customers = customer_sales_agg.withColumn("sales_rank", F.rank().over(Window.orderBy(F.col("total_cust_sales").desc())))
print("=== 34. Ranked Customers By Total Sales ===")
ranked_customers.show(10, truncate=False)

# 35. Top 3 products in every category
prod_cat = products.join(categories, "categoryid", "inner")
cat_window = Window.partitionBy("categoryid").orderBy(F.col("price").desc())
top_3_products_per_cat = prod_cat.withColumn("rn", F.row_number().over(cat_window)).filter("rn <= 3").drop("rn")
print("=== 35. Top 3 Products Per Category ===")
top_3_products_per_cat.select("categoryid", "categoryname", "productid", "productname", "price").show(15, truncate=False)

# 36. Most expensive product in every category
most_expensive_per_cat = prod_cat.withColumn("rn", F.row_number().over(cat_window)).filter("rn = 1").drop("rn")
print("=== 36. Most Expensive Product Per Category ===")
most_expensive_per_cat.select("categoryid", "categoryname", "productid", "productname", "price").show(10, truncate=False)

# 37. Latest shipment for every order
ship_window = Window.partitionBy("orderid").orderBy(F.col("shipdate").desc())
latest_shipments = shipments.withColumn("rn", F.row_number().over(ship_window)).filter("rn = 1").drop("rn")
print("=== 37. Latest Shipment Per Order ===")
latest_shipments.show(10, truncate=False)

print("Part 5 Window Functions Executed Successfully!")

=== 32. Latest Order Per Customer ===
+----------+-------+----------+----------+
|customerid|orderid|status    |orderdate |
+----------+-------+----------+----------+
|3         |17011  |PENDING   |2026-04-01|
|4         |21695  |PROCESSING|2025-10-15|
|6         |19312  |PENDING   |2026-07-03|
|9         |2861   |DELIVERED |2025-11-29|
|11        |33028  |PROCESSING|2026-01-18|
|14        |39870  |PENDING   |2026-03-13|
|17        |11899  |DELIVERED |2026-06-07|
|19        |18595  |PENDING   |2026-07-10|
|23        |11678  |CANCELLED |2026-07-02|
|28        |11271  |CANCELLED |2025-12-05|
+----------+-------+----------+----------+
only showing top 10 rows

=== 33. First Order Per Customer ===
+----------+-------+----------+----------+
|customerid|orderid|status    |orderdate |
+----------+-------+----------+----------+
|3         |12675  |CANCELLED |2024-01-08|
|4         |21695  |PROCESSING|2025-10-15|
|6         |30917  |SHIPPED   |2024-03-09|
|9         |30999  |DELIVERED |2024-02-

Cell 6: Part 6 & Part 11 - Building Data Models & Joins

In [27]:
# 38. Join orders + customers
orders_customers = orders.join(customers, "customerid", "inner").select(
    "orderid", "customerid", F.col("full_name").alias("customer_name"), "city", "country", "orderdate", "status"
)
print("=== 38. Orders + Customers Join ===")
orders_customers.show(10, truncate=False)

# 40. Join products + categories
products_categories = products.join(categories, "categoryid", "inner").select(
    "productid", "productname", "categoryid", "categoryname", "price"
)
print("=== 40. Products + Categories Join ===")
products_categories.show(10, truncate=False)

# 41. Join products + product_suppliers + suppliers
products_suppliers_join = products.join(product_suppliers, "productid", "inner") \
                                  .join(suppliers, "supplierid", "inner") \
                                  .select("productid", "productname", "supplierid", "suppliername", "country")
print("=== 41. Products + Suppliers Join ===")
products_suppliers_join.show(10, truncate=False)

# 42. Join orders + payments
orders_payments = orders.join(payments, "orderid", "inner").select(
    "orderid", "customerid", "orderdate", "paymentid", "paymentmethod", "amount"
)
print("=== 42. Orders + Payments Join ===")
orders_payments.show(10, truncate=False)

# 43. Join orders + shipments + shippers
orders_shipments = orders.join(shipments, "orderid", "inner") \
                          .join(shippers, "shipperid", "inner") \
                          .select("orderid", "customerid", "shipmentid", "companyname", "shipdate", "deliverydate")
print("=== 43. Orders + Shipments + Shippers Join ===")
orders_shipments.show(10, truncate=False)

# 39 & Part 11: Final fact_sales Table
fact_sales = orders.join(order_details, "orderid", "inner") \
                   .join(products.select("productid", "categoryid"), "productid", "inner") \
                   .select(
                       "orderid", "customerid", "productid", "categoryid",
                       "orderdate", "quantity", "unitprice", "total_amount", "status",
                       "year", "month"
                   )

print("=== 39 & Part 11: Fact Sales Table Sample ===")
fact_sales.show(10, truncate=False)

print("Part 6 & Part 11 Data Models & Joins Completed Successfully!")

=== 38. Orders + Customers Join ===
+-------+----------+----------------+-----+-------+----------+----------+
|orderid|customerid|customer_name   |city |country|orderdate |status    |
+-------+----------+----------------+-----+-------+----------+----------+
|37134  |1         |Danielle Johnson|Giza |Egypt  |2024-03-23|PROCESSING|
|35385  |1         |Danielle Johnson|Giza |Egypt  |2026-08-14|CANCELLED |
|34482  |1         |Danielle Johnson|Giza |Egypt  |2026-03-12|SHIPPED   |
|5073   |1         |Danielle Johnson|Giza |Egypt  |2024-01-01|SHIPPED   |
|36443  |2         |Joshua Walker   |Dubai|Jordan |2026-05-15|SHIPPED   |
|21249  |2         |Joshua Walker   |Dubai|Jordan |2026-03-28|PENDING   |
|3924   |2         |Joshua Walker   |Dubai|Jordan |2026-08-31|DELIVERED |
|17987  |2         |Joshua Walker   |Dubai|Jordan |2024-11-26|SHIPPED   |
|38543  |2         |Joshua Walker   |Dubai|Jordan |2026-05-19|PROCESSING|
|34899  |2         |Joshua Walker   |Dubai|Jordan |2024-11-06|DELIVERED |
+-

Cell 7: Part 7, 8, 9 - Aggregations & Advanced Analytics

In [28]:
# Part 7: Aggregated Datasets
customer_sales = fact_sales.groupBy("customerid").agg(
    F.sum("total_amount").alias("total_sales"), 
    F.countDistinct("orderid").alias("order_count")
)
print("=== Customer Sales Aggregation ===")
customer_sales.show(10, truncate=False)

product_sales = fact_sales.groupBy("productid").agg(
    F.sum("total_amount").alias("total_sales"), 
    F.sum("quantity").alias("total_quantity")
)
print("=== Product Sales Aggregation ===")
product_sales.show(10, truncate=False)

category_sales = fact_sales.groupBy("categoryid").agg(
    F.sum("total_amount").alias("total_sales")
)
print("=== Category Sales Aggregation ===")
category_sales.show(10, truncate=False)

monthly_sales = fact_sales.groupBy("year", "month").agg(
    F.sum("total_amount").alias("total_sales")
)
print("=== Monthly Sales Aggregation ===")
monthly_sales.orderBy("year", "month").show(10, truncate=False)

# Part 8: Top 10 Customers & Products
top_10_customers = customer_sales.orderBy(F.col("total_sales").desc()).limit(10)
print("=== Top 10 Customers by Sales ===")
top_10_customers.show(truncate=False)

top_10_products = product_sales.orderBy(F.col("total_sales").desc()).limit(10)
print("=== Top 10 Products by Sales ===")
top_10_products.show(truncate=False)

# Customers with no orders
customers_no_orders = customers.join(orders, "customerid", "left_anti")
print("Customers Who Never Placed an Order Count:", customers_no_orders.count())
customers_no_orders.select("customerid", "full_name", "email").show(5, truncate=False)

# Products with no orders
products_never_ordered = products.join(order_details, "productid", "left_anti")
print("Products That Were Never Ordered Count:", products_never_ordered.count())
products_never_ordered.select("productid", "productname", "price").show(5, truncate=False)

print("Part 7, 8, 9 Aggregations and Advanced Analytics Completed Successfully!")

=== Customer Sales Aggregation ===
+----------+-----------+-----------+
|customerid|total_sales|order_count|
+----------+-----------+-----------+
|3491      |255941.09  |9          |
|3576      |76078.77   |5          |
|4987      |67401.81   |2          |
|3752      |153305.69  |7          |
|4201      |55402.81   |4          |
|1479      |37329.75   |4          |
|4495      |30318.64   |4          |
|2872      |83724.41   |4          |
|7843      |136985.35  |3          |
|1650      |112276.75  |6          |
+----------+-----------+-----------+
only showing top 10 rows

=== Product Sales Aggregation ===
+---------+-----------+--------------+
|productid|total_sales|total_quantity|
+---------+-----------+--------------+
|660      |929102.55  |465           |
|633      |799029.95  |541           |
|93       |402434.21  |449           |
|70       |858383.46  |594           |
|692      |1615026.87 |753           |
|835      |1383119.25 |505           |
|644      |1409234.40 |504          

Cell 8: Part 10 - Save Outputs as Parquet + Snappy to S3

In [30]:
print("Starting to clean duplicate columns and write output datasets to S3...")

# Clean temporary join columns from orders if present
temp_cols = ["calc_order_total", "calc_order_qty"]
for col_name in temp_cols:
    if orders.columns.count(col_name) > 1:
        orders = orders.drop(col_name)

# Write core dimensions & facts to Parquet with Snappy compression
customers.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/customers/")
print("Saved: customers")

products.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/products/")
print("Saved: products")

categories.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/categories/")
print("Saved: categories")

order_details.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/order_details/")
print("Saved: order_details")

payments.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/payments/")
print("Saved: payments")

shipments.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/shipments/")
print("Saved: shipments")

# Write Aggregated Datasets
customer_sales.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/customer_sales/")
print("Saved: customer_sales")

product_sales.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/product_sales/")
print("Saved: product_sales")

category_sales.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/category_sales/")
print("Saved: category_sales")

monthly_sales.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/monthly_sales/")
print("Saved: monthly_sales")

fact_sales.write.mode("overwrite").parquet(f"{PROCESSED_PATH}/fact_sales/")
print("Saved: fact_sales")

# Partition Orders Dataset by year and month
orders.write.mode("overwrite").partitionBy("year", "month").parquet(f"{PROCESSED_PATH}/orders/")
print("Saved: orders (Partitioned by year and month)")

print("SUCCESS: All datasets successfully written to S3 in Parquet (Snappy) format with partitioning!")

Starting to clean duplicate columns and write output datasets to S3...
Saved: customers
Saved: products
Saved: categories
Saved: order_details
Saved: payments
Saved: shipments
Saved: customer_sales
Saved: product_sales
Saved: category_sales
Saved: monthly_sales
Saved: fact_sales
Saved: orders (Partitioned by year and month)
SUCCESS: All datasets successfully written to S3 in Parquet (Snappy) format with partitioning!
